In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('data/Sample - Superstore.csv.zip', encoding='latin-1')

In [4]:
print('Loaded: ',df.shape)

Loaded:  (9994, 21)


## Data ki copy

In [9]:
df_clean = df.copy()

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)
df_clean.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit']

## Text Columns Ke Extra Spaces Remove

In [13]:
text_cols = df_clean.select_dtypes(include='object').columns

for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

print('Spaces clean ho gaya..')

Spaces clean ho gaya..


## Date Coumns Ko Date Time Main Badalna

In [14]:
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
df_clean['ship_date'] = pd.to_datetime(df_clean['ship_date'])

df_clean[['order_date', 'ship_date']].dtypes

order_date    datetime64[ns]
ship_date     datetime64[ns]
dtype: object

## Postal code ko 5 digit string banao

In [17]:
df_clean['postal_code'] = df_clean['postal_code'].astype(str).str.zfill(5)

df_clean['postal_code'].head()

0    42420
1    42420
2    90036
3    33311
4    33311
Name: postal_code, dtype: object

In [20]:
print(df_clean['postal_code'].apply(type).unique())

[<class 'str'>]


## Duplicate Aur Missing Value

In [21]:
print('Duplicated Rows :',df_clean.duplicated().sum)

Duplicated Rows : <bound method Series.sum of 0       False
1       False
2       False
3       False
4       False
        ...  
9989    False
9990    False
9991    False
9992    False
9993    False
Length: 9994, dtype: bool>


In [22]:
print('Duplicated Row IDs :',df_clean['row_id'].duplicated().sum())

Duplicated Row IDs : 0


In [27]:
df_clean = df_clean.drop_duplicates()

print('Total missing values :',df_clean.isnull().sum().sum())
print('Rows after cleaning :',df_clean.shape[0])

Total missing values : 0
Rows after cleaning : 9994


## Validation (logic check)

In [31]:
print('Ship Date < Order Date:', (df_clean['ship_date'] < df_clean['order_date']).sum())
print('Sales <= 0            :', (df_clean['sales'] <=0).sum())
print('Quantity <= 0         :', (df_clean['quantity'] <= 0).sum())
print('Discount outside 0-1  :', ((df_clean['discount'] < 0) | (df_clean['discount'] > 1)).sum())

Ship Date < Order Date: 0
Sales <= 0            : 0
Quantity <= 0         : 0
Discount outside 0-1  : 0


## New Columns

In [32]:
df_clean['order_year'] = df_clean['order_date'].dt.year
df_clean['order_month'] = df_clean['order_date'].dt.month
df_clean['order_month_name'] = df_clean['order_date'].dt.month_name()
df_clean['order_quarter'] = df_clean['order_date'].dt.quarter

df_clean['shipping_days'] = (df_clean['ship_date'] - df_clean['order_date']).dt.days

df_clean['profit_margin_pct'] = (df_clean['profit'] / df_clean['sales'] * 100).round(2)

# 1 = loss wala order, 0 = profit wala order
df_clean['is_loss'] = (df_clean['profit'] < 0).astype(int)

df_clean[['order_date', 'order_year', 'order_quarter', 'shipping_days', 'profit_margin_pct', 'is_loss']].head()

,order_date,order_year,order_quarter,shipping_days,profit_margin_pct,is_loss
0,2016-11-08,2016,4,3,16.00,0
1,2016-11-08,2016,4,3,30.00,0
2,2016-06-12,2016,2,4,47.00,0
3,2015-10-11,2015,4,7,-40.00,1
4,2015-10-11,2015,4,7,11.25,0


## New Columns Ka Quick Check

In [35]:
print(df_clean['shipping_days'].describe())
print('\nLoss Walw Orders: ',df_clean['is_loss'].sum())
print('Order Per Year:')
print(df_clean['order_year'].value_counts().sort_index())

count    9994.000000
mean        3.958175
std         1.747567
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: shipping_days, dtype: float64

Loss Walw Orders:  1871
Order Per Year:
order_year
2014    1993
2015    2102
2016    2587
2017    3312
Name: count, dtype: int64


## Final Check + cleaned file save

In [36]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   row_id             9994 non-null   int64         
 1   order_id           9994 non-null   object        
 2   order_date         9994 non-null   datetime64[ns]
 3   ship_date          9994 non-null   datetime64[ns]
 4   ship_mode          9994 non-null   object        
 5   customer_id        9994 non-null   object        
 6   customer_name      9994 non-null   object        
 7   segment            9994 non-null   object        
 8   country            9994 non-null   object        
 9   city               9994 non-null   object        
 10  state              9994 non-null   object        
 11  postal_code        9994 non-null   object        
 12  region             9994 non-null   object        
 13  product_id         9994 non-null   object        
 14  category

In [38]:
df_clean.to_csv('data/super_store_clean.csv', index=False)
print('\nCleaned file save ho gyi : superstore_clean.csv')


Cleaned file save ho gyi : superstore_clean.csv
